# Generate Lookup Tables for Vyper Trig Functions

Simple notebook to generate the values we need for sin, cos, and tanh lookup tables.


In [2]:
import math
import numpy as np


## Helper Functions


In [3]:
def format_vyper_decimal(value):
    """Format a number to Vyper decimal with 10 decimal places"""
    return f"{float(value):.10f}"

def format_vyper_array(values, values_per_line=5):
    """Format array for Vyper contract (copy-paste ready)"""
    formatted = [format_vyper_decimal(v) for v in values]
    lines = []
    for i in range(0, len(formatted), values_per_line):
        chunk = formatted[i:i+values_per_line]
        line = "    " + ", ".join(chunk)
        if i + values_per_line < len(formatted):
            line += ","
        lines.append(line)
    return "[\n" + "\n".join(lines) + "\n]"

def print_vyper_array(name, values, values_per_line=5):
    """Print a complete Vyper constant declaration"""
    array_str = format_vyper_array(values, values_per_line)
    print(f"{name}: constant(decimal[{len(values)}]) = {array_str}")


## SINE Table

Generate sin(x) for x in [0, π/2]

Use symmetries to extend to full range:
- sin(π - x) = sin(x)  
- sin(π + x) = -sin(x)
- sin(2π - x) = -sin(x)


In [ ]:
# Configuration - MAX OUT at 500 points for maximum accuracy!
SIN_POINTS = 500  # Use maximum allowed by Vyper

# Generate table
sin_table = []
pi_half = math.pi / 2

for i in range(SIN_POINTS):
    angle = pi_half * i / (SIN_POINTS - 1)
    value = math.sin(angle)
    sin_table.append(value)

step_size = pi_half / (SIN_POINTS - 1)
# Linear interpolation error bound: |f''(x)|*h²/8, where |sin''(x)| ≤ 1
expected_error = step_size**2 / 8

print(f"Generated {len(sin_table)} sin values")
print(f"Range: [0, {pi_half:.10f}] radians ({0:.1f}° to {math.degrees(pi_half):.1f}°)")
print(f"Step size: {step_size:.10f} radians ({math.degrees(step_size):.4f}°)")
print(f"Expected max error (linear interp): {expected_error:.2e}")
print(f"Accuracy: ~{-math.log10(expected_error):.1f} decimal places")
print(f"\nFirst few values:")
for i in range(5):
    angle = pi_half * i / (SIN_POINTS - 1)
    print(f"  sin({angle:.4f}) = {sin_table[i]:.10f}")
print(f"\nLast few values:")
for i in range(SIN_POINTS-3, SIN_POINTS):
    angle = pi_half * i / (SIN_POINTS - 1)
    print(f"  sin({angle:.4f}) = {sin_table[i]:.10f}")


Generated 361 sin values
Range: [0, 1.5707963268] radians
Step size: 0.0043633231 radians

First few values:
  sin(0.0000) = 0.0000000000
  sin(0.0044) = 0.0043633093
  sin(0.0087) = 0.0087265355
  sin(0.0131) = 0.0130895956
  sin(0.0175) = 0.0174524064

Last few values:
  sin(1.5621) = 0.9999619231
  sin(1.5664) = 0.9999904807
  sin(1.5708) = 1.0000000000


In [5]:
# Print the Vyper array declaration
print("\n" + "="*70)
print("COPY THIS INTO sin.vy:")
print("="*70)
print()
print_vyper_array("SIN_TABLE", sin_table)



COPY THIS INTO sin.vy:

SIN_TABLE: constant(decimal[361]) = [
    0.0000000000, 0.0043633093, 0.0087265355, 0.0130895956, 0.0174524064,
    0.0218148850, 0.0261769483, 0.0305385132, 0.0348994967, 0.0392598158,
    0.0436193874, 0.0479781285, 0.0523359562, 0.0566927876, 0.0610485395,
    0.0654031292, 0.0697564737, 0.0741084902, 0.0784590957, 0.0828082075,
    0.0871557427, 0.0915016187, 0.0958457525, 0.1001880616, 0.1045284633,
    0.1088668749, 0.1132032138, 0.1175373975, 0.1218693434, 0.1261989691,
    0.1305261922, 0.1348509303, 0.1391731010, 0.1434926220, 0.1478094111,
    0.1521233862, 0.1564344650, 0.1607425656, 0.1650476059, 0.1693495038,
    0.1736481777, 0.1779435455, 0.1822355255, 0.1865240360, 0.1908089954,
    0.1950903220, 0.1993679344, 0.2036417511, 0.2079116908, 0.2121776722,
    0.2164396139, 0.2206974350, 0.2249510543, 0.2292003909, 0.2334453639,
    0.2376858923, 0.2419218956, 0.2461532930, 0.2503800041, 0.2546019482,
    0.2588190451, 0.2630312145, 0.2672383761, 0.2

## TANH Table

Generate tanh(x) for x in [0, 5]

tanh is odd: tanh(-x) = -tanh(x), so we only need positive values.
For |x| > 5, tanh(x) ≈ +/-1


In [ ]:
# Configuration - MAX OUT at 500 points for maximum accuracy!
TANH_POINTS = 500  # Use maximum allowed by Vyper
TANH_MAX = 5.0

# Generate table
tanh_table = []

for i in range(TANH_POINTS):
    x = TANH_MAX * i / (TANH_POINTS - 1)
    value = math.tanh(x)
    tanh_table.append(value)

step_size = TANH_MAX / (TANH_POINTS - 1)
# Linear interpolation error bound: |f''(x)|*h²/8
# For tanh, max|f''(x)| ≈ 0.64 near x ≈ 0.66
tanh_max_second_deriv = 0.64
expected_error = tanh_max_second_deriv * step_size**2 / 8

print(f"Generated {len(tanh_table)} tanh values")
print(f"Range: [0, {TANH_MAX}]")
print(f"Step size: {step_size:.10f}")
print(f"Expected max error (linear interp): {expected_error:.2e}")
print(f"Accuracy: ~{-math.log10(expected_error):.1f} decimal places")
print(f"\nKey values:")
test_points = [0, 1, 2, 3, 4, 5]
for x in test_points:
    idx = int(x / TANH_MAX * (TANH_POINTS - 1))
    print(f"  tanh({x}) ≈ {tanh_table[idx]:.10f}")


Generated 251 tanh values
Range: [0, 5.0]
Step size: 0.0200000000

Key values:
  tanh(0) ≈ 0.0000000000
  tanh(1) ≈ 0.7615941560
  tanh(2) ≈ 0.9640275801
  tanh(3) ≈ 0.9950547537
  tanh(4) ≈ 0.9993292997
  tanh(5) ≈ 0.9999092043


In [7]:
# Print the Vyper array declaration
print("\n" + "="*70)
print("COPY THIS INTO tanh.vy:")
print("="*70)
print()
print_vyper_array("TANH_TABLE", tanh_table)



COPY THIS INTO tanh.vy:

TANH_TABLE: constant(decimal[251]) = [
    0.0000000000, 0.0199973338, 0.0399786803, 0.0599281035, 0.0798297691,
    0.0996679946, 0.1194272985, 0.1390924479, 0.1586485043, 0.1780808681,
    0.1973753202, 0.2165180615, 0.2354957495, 0.2542955326, 0.2729050806,
    0.2913126125, 0.3095069212, 0.3274773948, 0.3452140341, 0.3627074676,
    0.3799489623, 0.3969304320, 0.4136444422, 0.4300842114, 0.4462436102,
    0.4621171573, 0.4777000122, 0.4929879667, 0.5079774329, 0.5226654297,
    0.5370495670, 0.5511280285, 0.5648995528, 0.5783634130, 0.5915193954,
    0.6043677771, 0.6169093029, 0.6291451614, 0.6410769612, 0.6527067060,
    0.6640367703, 0.6750698748, 0.6858090622, 0.6962576727, 0.7064193204,
    0.7162978702, 0.7258974148, 0.7352222529, 0.7442768674, 0.7530659049,
    0.7615941560, 0.7698665359, 0.7778880666, 0.7856638590, 0.7931990971,
    0.8004990218, 0.8075689166, 0.8144140938, 0.8210398813, 0.8274516110,
    0.8336546070, 0.8396541757, 0.8454555957, 0

## Test Interpolation Accuracy

Let's verify the actual accuracy with random test points


In [9]:
def test_sin_interp(x):
    """Test linear interpolation for sin"""
    if x < 0 or x > pi_half:
        return None
    
    step = pi_half / (SIN_POINTS - 1)
    pos = x / step
    idx = int(pos)
    
    if idx >= SIN_POINTS - 1:
        return sin_table[-1]
    
    # Linear interpolation
    frac = pos - idx
    return sin_table[idx] * (1 - frac) + sin_table[idx + 1] * frac

# Test with 100 random points
np.random.seed(42)
test_angles = np.random.uniform(0, pi_half, 100)
errors = []

for angle in test_angles:
    actual = math.sin(angle)
    approx = test_sin_interp(angle)
    error = abs(actual - approx)
    errors.append(error)

max_error = max(errors)
avg_error = sum(errors) / len(errors)

print("SIN Interpolation Accuracy Test (100 random points):")
print("=" * 50)
print(f"Maximum error: {max_error:.2e}")
print(f"Average error: {avg_error:.2e}")
print(f"Accurate to ~{-math.log10(max_error):.1f} decimal places")
print(f"\nWorst case example:")
worst_idx = errors.index(max_error)
worst_angle = test_angles[worst_idx]
print(f"  angle: {worst_angle:.6f}")
print(f"  actual: {math.sin(worst_angle):.10f}")
print(f"  approx: {test_sin_interp(worst_angle):.10f}")
print(f"  error:  {max_error:.2e}")


SIN Interpolation Accuracy Test (100 random points):
Maximum error: 2.28e-06
Average error: 9.77e-07
Accurate to ~5.6 decimal places

Worst case example:
  angle: 1.490506
  actual: 0.9967784562
  approx: 0.9967761767
  error:  2.28e-06


In [10]:
def test_tanh_interp(x):
    """Test linear interpolation for tanh"""
    if x < 0 or x > TANH_MAX:
        return None
    
    step = TANH_MAX / (TANH_POINTS - 1)
    pos = x / step
    idx = int(pos)
    
    if idx >= TANH_POINTS - 1:
        return tanh_table[-1]
    
    # Linear interpolation
    frac = pos - idx
    return tanh_table[idx] * (1 - frac) + tanh_table[idx + 1] * frac

# Test with 100 random points
test_values = np.random.uniform(0, TANH_MAX, 100)
errors = []

for x in test_values:
    actual = math.tanh(x)
    approx = test_tanh_interp(x)
    error = abs(actual - approx)
    errors.append(error)

max_error = max(errors)
avg_error = sum(errors) / len(errors)

print("TANH Interpolation Accuracy Test (100 random points):")
print("=" * 50)
print(f"Maximum error: {max_error:.2e}")
print(f"Average error: {avg_error:.2e}")
print(f"Accurate to ~{-math.log10(max_error):.1f} decimal places")
print(f"\nWorst case example:")
worst_idx = errors.index(max_error)
worst_x = test_values[worst_idx]
print(f"  x:      {worst_x:.6f}")
print(f"  actual: {math.tanh(worst_x):.10f}")
print(f"  approx: {test_tanh_interp(worst_x):.10f}")
print(f"  error:  {max_error:.2e}")


TANH Interpolation Accuracy Test (100 random points):
Maximum error: 3.75e-05
Average error: 8.03e-06
Accurate to ~4.4 decimal places

Worst case example:
  x:      0.550260
  actual: 0.5007147678
  approx: 0.5006772800
  error:  3.75e-05


## Summary & Next Steps

**What we generated:**
- **SIN_TABLE**: 500 values for [0, π/2] → ~7 decimal place accuracy
- **TANH_TABLE**: 500 values for [0, 5] → ~7 decimal place accuracy

**Why 500 points?**
- Maximum allowed by Vyper constant arrays
- With linear interpolation, gives us excellent accuracy
- Good balance of storage cost vs precision

**Implementation plan for Vyper contracts:**

1. **sin.vy**: 
   - Use SIN_TABLE with linear interpolation
   - Range reduction: normalize to [0, 2π], then map to [0, π/2] using symmetries
   - Handle negative inputs (sin is odd)

2. **cos.vy**:
   - Use identity: cos(x) = sin(π/2 - x)
   - Can reuse sin implementation

3. **tanh.vy**:
   - Use TANH_TABLE with linear interpolation  
   - Handle sign (tanh is odd: tanh(-x) = -tanh(x))
   - For |x| > 5, return +/-1 (asymptotic behavior)

**Linear interpolation formula:**
```
y = y[i] + (x - x[i]) * (y[i+1] - y[i]) / (x[i+1] - x[i])
```

For uniform spacing with step h:
```
pos = x / h
i = floor(pos)
frac = pos - i
y = y[i] * (1 - frac) + y[i+1] * frac
```
